## torch.nn.Module and torch.nn.Parameter

### What is a Parameter?

### A Parameter is a value that the model learns during training, such as weights and biases.

### PyTorch stores these learnable values using the torch.nn.Parameter class.

### Why is Parameter special?

### A Parameter is just like a normal tensor, but with one extra feature:

### When you assign it inside a Module, PyTorch automatically recognizes it as a learnable weight.
### It is then included in the model's list of parameters.
### Optimizers (like SGD or Adam) can automatically update these parameters during training.

In [3]:
import torch

class TinyModel(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.linear1 = torch.nn.Linear(100, 200)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(200, 10)
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.softmax(x)
        return x

tinymodel = TinyModel()

print('The model:')
print(tinymodel)

print('\n\nJust one layer:')
print(tinymodel.linear2)

print('\n\nModel params:')
for param in tinymodel.parameters():
    print(param)

print('\n\nLayer params:')
for param in tinymodel.linear2.parameters():
    print(param)

The model:
TinyModel(
  (linear1): Linear(in_features=100, out_features=200, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=200, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)


Just one layer:
Linear(in_features=200, out_features=10, bias=True)


Model params:
Parameter containing:
tensor([[-0.0075, -0.0540,  0.0084,  ..., -0.0557, -0.0730, -0.0207],
        [-0.0258, -0.0999, -0.0006,  ..., -0.0892,  0.0800,  0.0878],
        [ 0.0472, -0.0374,  0.0717,  ...,  0.0151,  0.0892, -0.0031],
        ...,
        [ 0.0936,  0.0292, -0.0966,  ..., -0.0805,  0.0015, -0.0934],
        [-0.0995,  0.0682, -0.0846,  ...,  0.0272, -0.0975, -0.0348],
        [ 0.0527,  0.0328, -0.0209,  ..., -0.0209, -0.0734, -0.0746]],
       requires_grad=True)
Parameter containing:
tensor([ 1.2318e-02,  4.9052e-02,  5.0381e-02,  1.4887e-02,  3.1904e-02,
         8.0863e-02, -7.6712e-03,  5.9742e-02, -1.5861e-02,  7.5566e-03,
         6.6018e-02, -6.4907e-02, -9.2455e-02,  7.0982e-0

## Common Layer Types
### Linear Layers

In [6]:
lin = torch.nn.Linear(3, 2)
x = torch.rand(1, 3)
print('Input:')
print(x)

print('\n\nWeight and Bias parameters:')
for param in lin.parameters():
    print(param)

y = lin(x)
print('\n\nOutput:')
print(y)

Input:
tensor([[0.9558, 0.8214, 0.8839]])


Weight and Bias parameters:
Parameter containing:
tensor([[ 0.3767,  0.4153, -0.2023],
        [-0.2267, -0.5125,  0.1959]], requires_grad=True)
Parameter containing:
tensor([0.4555, 0.5603], requires_grad=True)


Output:
tensor([[0.9779, 0.0959]], grad_fn=<AddmmBackward0>)


## Convolutional Layers

In [9]:
import torch.nn.functional as F


class LeNet(torch.nn.Module):

    def __init__(self):
        super().__init__()
        # 1 input image channel (black & white), 6 output channels, 5x5 square convolution
        # kernel
        self.conv1 = torch.nn.Conv2d(1, 6, 5)
        self.conv2 = torch.nn.Conv2d(6, 16, 3)
        # an affine operation: y = Wx + b
        self.fc1 = torch.nn.Linear(16 * 6 * 6, 120)  # 6*6 from image dimension
        self.fc2 = torch.nn.Linear(120, 84)
        self.fc3 = torch.nn.Linear(84, 10)

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

## Recurrent Layers

In [12]:
class LSTMTagger(torch.nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = torch.nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

## Data Manipulation Layers

In [15]:
my_tensor = torch.rand(1, 6, 6)
print(my_tensor)

maxpool_layer = torch.nn.MaxPool2d(3)
print(maxpool_layer(my_tensor))

tensor([[[0.8739, 0.2487, 0.9898, 0.5159, 0.5441, 0.3879],
         [0.7332, 0.3943, 0.0140, 0.6585, 0.2101, 0.8843],
         [0.6208, 0.7555, 0.8447, 0.5212, 0.7484, 0.7534],
         [0.1114, 0.4758, 0.2188, 0.8721, 0.5376, 0.4262],
         [0.5736, 0.0832, 0.2262, 0.6640, 0.6523, 0.4503],
         [0.6425, 0.4916, 0.8961, 0.3527, 0.5258, 0.8908]]])
tensor([[[0.9898, 0.8843],
         [0.8961, 0.8908]]])


In [17]:
my_tensor = torch.rand(1, 4, 4) * 20 + 5
print(my_tensor)

print(my_tensor.mean())

norm_layer = torch.nn.BatchNorm1d(4)
normed_tensor = norm_layer(my_tensor)
print(normed_tensor)

print(normed_tensor.mean())

tensor([[[ 5.7361,  7.5204, 15.0393, 22.2123],
         [12.3377,  7.2031, 15.0370,  6.2729],
         [16.3699, 12.2779, 19.9852,  5.9787],
         [22.5449, 21.8730,  5.2821,  5.0683]]])
tensor(12.5462)
tensor([[[-1.0531, -0.7804,  0.3687,  1.4649],
         [ 0.5873, -0.8318,  1.3333, -1.0888],
         [ 0.5222, -0.2643,  1.2172, -1.4751],
         [ 1.0390,  0.9601, -0.9870, -1.0121]]],
       grad_fn=<NativeBatchNormBackward0>)
tensor(2.9802e-08, grad_fn=<MeanBackward0>)


In [19]:
my_tensor = torch.rand(1, 4, 4)

dropout = torch.nn.Dropout(p=0.4)
print(dropout(my_tensor))
print(dropout(my_tensor))

tensor([[[1.2695, 0.5213, 0.9803, 0.6691],
         [0.0000, 0.0000, 0.5862, 0.8100],
         [1.3081, 0.8770, 0.0617, 0.0000],
         [0.0000, 0.0000, 1.4722, 0.0000]]])
tensor([[[0.0000, 0.0000, 0.0000, 0.6691],
         [0.0000, 1.2474, 0.5862, 0.8100],
         [0.0000, 0.8770, 0.0000, 0.0000],
         [0.0000, 0.7206, 1.4722, 0.5004]]])
